# Week 2 – Day 10: Baseline Model Evaluation
**Project:** DubaiEstate AI 360 – Agentic AI Powered Real Estate Investment  
**Company:** NEOSKILLZ  
**Intern:** Nitesh Kumar Barnwal  
**Domain:** Artificial Intelligence & Machine Learning (AI/ML)  
**Branch:** `feature/week02-aiml-nitesh`  

---

## 1. Objective
Establish a reliable, reproducible, and leakage-free benchmark for Dubai real estate transaction value prediction (`TRANS_VALUE`) by:
1. Reusing the Day 8 dataset loader (`model_dataset.py`) and Day 9 preprocessing pipeline (`preprocess.py`).
2. Creating a date-aware chronological 80% / 20% train-test split without lookahead bias.
3. Training a naive `DummyRegressor` benchmark (median and mean).
4. Training a simple regression baseline using `LinearRegression`.
5. Evaluating both models using **MAE** and **RMSE** on the untouched test partition.
6. Exporting actual evaluation results to `metrics.json`.

In [1]:
import json
from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

# Add week02 directory to sys.path for local module resolution
CURRENT_DIR = Path.cwd()
if str(CURRENT_DIR) not in sys.path:
    sys.path.insert(0, str(CURRENT_DIR))

from model_dataset import load_transactions
from preprocess import (
    ALL_FEATURES,
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES,
    TARGET_COLUMN,
    build_preprocessor,
    check_transformed_matrix,
    fit_preprocessor,
    get_candidate_sales,
    prepare_features,
    transform_features,
)

print("All required libraries and week02 modules imported successfully.")

All required libraries and week02 modules imported successfully.


## 2. Dataset Loading & Arm's-Length Transaction Isolation
Load official Dubai real estate transactions and isolate valid arm's-length sale records (`PROCEDURE_EN` matching `sell` or `sale`, and `TRANS_VALUE > 0`) using Day 8 and Day 9 logic.

In [2]:
# Load raw transaction records across both CSV files
raw_df = load_transactions()
print(f"Raw dataset volume: {len(raw_df):,} records, {raw_df.shape[1]} columns")

# Isolate arm's-length sales
sales_df = get_candidate_sales(raw_df)
print(f"Eligible candidate sale records: {len(sales_df):,} records")
print(f"Excluded non-sale/administrative records: {len(raw_df) - len(sales_df):,} records")

Raw dataset volume: 134,185 records, 22 columns
Eligible candidate sale records: 131,283 records
Excluded non-sale/administrative records: 2,902 records


## 3. Date-Aware Chronological Splitting (80% Train / 20% Test)
To replicate a production real estate valuation deployment and prevent lookahead bias:
- Transactions are ordered chronologically by `INSTANCE_DATE`.
- **Training Set (80%):** Past transactions used strictly to learn preprocessor statistics and model weights.
- **Test Set (20%):** Future unseen transactions held untouched for honest out-of-sample evaluation.

In [3]:
# Parse timestamps and sort chronologically
sales_df["INSTANCE_DATE"] = pd.to_datetime(sales_df["INSTANCE_DATE"], errors="coerce")
sales_df = sales_df.sort_values("INSTANCE_DATE").reset_index(drop=True)

# 80% / 20% chronological split
split_idx = int(len(sales_df) * 0.80)
train_df = sales_df.iloc[:split_idx].copy()
test_df = sales_df.iloc[split_idx:].copy()

print(f"Training set records: {len(train_df):,} ({len(train_df)/len(sales_df)*100:.1f}%)")
print(f"  Training Date Range: {train_df['INSTANCE_DATE'].min()} to {train_df['INSTANCE_DATE'].max()}")
print(f"Test set records:     {len(test_df):,} ({len(test_df)/len(sales_df)*100:.1f}%)")
print(f"  Test Date Range:     {test_df['INSTANCE_DATE'].min()} to {test_df['INSTANCE_DATE'].max()}")

assert train_df['INSTANCE_DATE'].max() <= test_df['INSTANCE_DATE'].min(), "Temporal split violated!"

Training set records: 105,026 (80.0%)
  Training Date Range: 2026-01-01 16:21:33 to 2026-08-18 14:17:11
Test set records:     26,257 (20.0%)
  Test Date Range:     2026-08-18 14:17:25 to 2026-10-29 13:59:21


## 4. Leakage-Free Feature Extraction & Preprocessing
Extract the 13 approved features (`2` numeric, `11` categorical) while systematically excluding `TRANS_VALUE` (target) and `TRANSACTION_NUMBER` (identifier).
Fit the scikit-learn `ColumnTransformer` **strictly on `X_train`** and transform `X_test` without refitting.

In [4]:
# Separate features and target
X_train, y_train = prepare_features(train_df, return_target=True)
X_test, y_test = prepare_features(test_df, return_target=True)

print(f"X_train shape: {X_train.shape} | Target in X_train? {TARGET_COLUMN in X_train.columns}")
print(f"X_test shape:  {X_test.shape}  | Target in X_test?  {TARGET_COLUMN in X_test.columns}")

# Build and fit preprocessor strictly on training data
preprocessor = build_preprocessor(sparse_output=True)
fit_preprocessor(preprocessor, X_train)

# Transform features
X_train_trans = transform_features(preprocessor, X_train)
X_test_trans = transform_features(preprocessor, X_test)

print(f"Transformed Training Matrix: {X_train_trans.shape} (Sparse CSR)")
print(f"Transformed Test Matrix:     {X_test_trans.shape} (Sparse CSR)")

# Numerical integrity checks
train_check = check_transformed_matrix(X_train_trans, "Training Matrix")
test_check = check_transformed_matrix(X_test_trans, "Test Matrix")
print(f"NaNs in Train: {train_check['nan_count']} | NaNs in Test: {test_check['nan_count']}")
print(f"Infs in Train: {train_check['inf_count']} | Infs in Test: {test_check['inf_count']}")

X_train shape: (105026, 13) | Target in X_train? False
X_test shape:  (26257, 13)  | Target in X_test?  False


Transformed Training Matrix: (105026, 375) (Sparse CSR)
Transformed Test Matrix:     (26257, 375) (Sparse CSR)
NaNs in Train: 0 | NaNs in Test: 0
Infs in Train: 0 | Infs in Test: 0


## 5. Baseline Model 1: DummyRegressor (Naive Benchmark)
The `DummyRegressor` serves as the non-informative heuristic benchmark. We evaluate:
1. **Median Strategy:** Constant prediction equal to the training target median.
2. **Mean Strategy:** Constant prediction equal to the training target mean.

In [5]:
# 1. Median Dummy Regressor
dummy_median = DummyRegressor(strategy="median")
dummy_median.fit(X_train_trans, y_train)
y_pred_dummy_median = dummy_median.predict(X_test_trans)

mae_dummy_median = mean_absolute_error(y_test, y_pred_dummy_median)
rmse_dummy_median = root_mean_squared_error(y_test, y_pred_dummy_median)

print("=== DUMMY REGRESSOR (MEDIAN) ===")
print(f"Predicted Constant: AED {dummy_median.constant_[0][0]:,.2f}")
print(f"Test MAE:           AED {mae_dummy_median:,.2f}")
print(f"Test RMSE:          AED {rmse_dummy_median:,.2f}")

# 2. Mean Dummy Regressor
dummy_mean = DummyRegressor(strategy="mean")
dummy_mean.fit(X_train_trans, y_train)
y_pred_dummy_mean = dummy_mean.predict(X_test_trans)

mae_dummy_mean = mean_absolute_error(y_test, y_pred_dummy_mean)
rmse_dummy_mean = root_mean_squared_error(y_test, y_pred_dummy_mean)

print("\n=== DUMMY REGRESSOR (MEAN) ===")
print(f"Predicted Constant: AED {dummy_mean.constant_[0][0]:,.2f}")
print(f"Test MAE:           AED {mae_dummy_mean:,.2f}")
print(f"Test RMSE:          AED {rmse_dummy_mean:,.2f}")

=== DUMMY REGRESSOR (MEDIAN) ===
Predicted Constant: AED 1,450,000.00
Test MAE:           AED 1,884,136.43
Test RMSE:          AED 9,815,454.11

=== DUMMY REGRESSOR (MEAN) ===
Predicted Constant: AED 3,138,378.77
Test MAE:           AED 2,547,931.28
Test RMSE:          AED 9,739,962.56


## 6. Baseline Model 2: Simple Linear Regression
Fit an unregularized Ordinary Least Squares (`LinearRegression`) estimator on `(X_train_trans, y_train)` and evaluate on the identical untouched test set.

In [6]:
lr_model = LinearRegression()
t_lr0 = time.time()
lr_model.fit(X_train_trans, y_train)
t_lr_fit = time.time() - t_lr0

y_pred_lr = lr_model.predict(X_test_trans)

mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = root_mean_squared_error(y_test, y_pred_lr)

print(f"Linear Regression fit completed in {t_lr_fit:.2f} seconds.")
print("=== LINEAR REGRESSION (OLS) ===")
print(f"Test MAE:  AED {mae_lr:,.2f}")
print(f"Test RMSE: AED {rmse_lr:,.2f}")

Linear Regression fit completed in 2.57 seconds.
=== LINEAR REGRESSION (OLS) ===
Test MAE:  AED 32,295,767.40
Test RMSE: AED 184,340,582.93


## 7. Baseline Model Comparison & Analysis
Compare both baselines side-by-side using the exact same untouched test records.

In [7]:
comparison_df = pd.DataFrame({
    "Model": ["DummyRegressor (Median)", "DummyRegressor (Mean)", "LinearRegression (OLS)"],
    "Test MAE (AED)": [mae_dummy_median, mae_dummy_mean, mae_lr],
    "Test RMSE (AED)": [rmse_dummy_median, rmse_dummy_mean, rmse_lr],
})

print(comparison_df.to_string(index=False))

print("\nBaseline Analysis & Findings:")
print("1. The Naive Median Regressor achieves a Test MAE of ~1.88M AED, establishing the true baseline to beat.")
print("2. Unregularized Linear Regression experiences severe variance inflation (MAE ~32.3M AED) on novel test transactions due to multicollinear dummy variables and unconstrained weights on rare categorical combinations.")
print("3. This demonstrates the critical need for regularized estimators or tree-based non-linear candidate models (e.g., Random Forest) in Day 11.")

                  Model  Test MAE (AED)  Test RMSE (AED)
DummyRegressor (Median)    1.884136e+06     9.815454e+06
  DummyRegressor (Mean)    2.547931e+06     9.739963e+06
 LinearRegression (OLS)    3.229577e+07     1.843406e+08

Baseline Analysis & Findings:
1. The Naive Median Regressor achieves a Test MAE of ~1.88M AED, establishing the true baseline to beat.
2. Unregularized Linear Regression experiences severe variance inflation (MAE ~32.3M AED) on novel test transactions due to multicollinear dummy variables and unconstrained weights on rare categorical combinations.
3. This demonstrates the critical need for regularized estimators or tree-based non-linear candidate models (e.g., Random Forest) in Day 11.


## 8. Exporting Evaluation Results to `metrics.json`
Save all verified evaluation results to `metrics.json` following the required schema.

In [8]:
metrics_payload = {
    "project": "DubaiEstate AI 360 – Agentic AI Powered Real Estate Investment",
    "intern": "Nitesh Kumar Barnwal",
    "domain": "AI/ML",
    "milestone": "Week 2 – Day 10: Baseline Model Evaluation",
    "branch": "feature/week02-aiml-nitesh",
    "target": TARGET_COLUMN,
    "dataset": {
        "total_eligible_records": int(len(sales_df)),
        "train_records": int(len(train_df)),
        "test_records": int(len(test_df)),
        "split_strategy": "chronological",
        "train_fraction": 0.80,
        "test_fraction": 0.20,
        "train_date_range": {
            "start": str(train_df['INSTANCE_DATE'].min()),
            "end": str(train_df['INSTANCE_DATE'].max())
        },
        "test_date_range": {
            "start": str(test_df['INSTANCE_DATE'].min()),
            "end": str(test_df['INSTANCE_DATE'].max())
        }
    },
    "features": {
        "numeric": NUMERIC_FEATURES,
        "categorical": CATEGORICAL_FEATURES,
        "total_input_features": len(ALL_FEATURES),
        "transformed_feature_dimension": int(X_train_trans.shape[1])
    },
    "models": {
        "DummyRegressor_median": {
            "strategy": "median",
            "predicted_constant": float(dummy_median.constant_[0][0]),
            "MAE": float(mae_dummy_median),
            "RMSE": float(rmse_dummy_median)
        },
        "DummyRegressor_mean": {
            "strategy": "mean",
            "predicted_constant": float(dummy_mean.constant_[0][0]),
            "MAE": float(mae_dummy_mean),
            "RMSE": float(rmse_dummy_mean)
        },
        "LinearRegression_OLS": {
            "fit_intercept": True,
            "MAE": float(mae_lr),
            "RMSE": float(rmse_lr)
        }
    },
    "summary": {
        "primary_baseline_MAE": float(mae_dummy_median),
        "primary_baseline_RMSE": float(rmse_dummy_median),
        "status": "Baseline established successfully; awaiting Day 11 candidate model comparison."
    }
}

metrics_path = Path("metrics.json")
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=4)

print(f"Successfully saved Day 10 evaluation metrics to: {metrics_path.resolve()}")

Successfully saved Day 10 evaluation metrics to: C:\Users\nites\Desktop\DubaiEstate-AI-360-Agentic-AI-Powered-Real-Estate-Investment\AI-ML\AI-ML\week02\metrics.json
